In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

from sklearn.ensemble import IsolationForest
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report

import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings(action='ignore')

In [ ]:
from google.colab import drive

drive.mount('/content/gdrive/')

path = '/content/gdrive/MyDrive/open'
test = '/content/gdrive/MyDrive/open/test.csv'
train = '/content/gdrive/MyDrive/open/train.csv'
val = '/content/gdrive/MyDrive/open/val.csv'

train = pd.read_csv(train)  # train.csv 읽기
val= pd.read_csv(val)      # val.csv 읽기
test= pd.read_csv(test)    # test.csv 읽기

display(train.head(), val.head(), test.head())


Mounted at /content/gdrive/


,ID,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,V29,V30
0,3,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,4.983721,-0.994972
1,4,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,1.418291,-0.994972
2,6,-0.425966,0.960523,1.141109,-0.168252,0.420987,-0.029728,0.476201,0.260314,-0.568671,...,-0.208254,-0.559825,-0.026398,-0.371427,-0.232794,0.105915,0.253844,0.081080,-0.256131,-0.994960
3,8,-0.644269,1.417964,1.074380,-0.492199,0.948934,0.428118,1.120631,-3.807864,0.615375,...,1.943465,-1.015455,0.057504,-0.649709,-0.415267,-0.051634,-1.206921,-1.085339,0.262698,-0.994901
4,9,-0.894286,0.286157,-0.113192,-0.271526,2.669599,3.721818,0.370145,0.851084,-0.392048,...,-0.073425,-0.268092,-0.204233,1.011592,0.373205,-0.384157,0.011747,0.142404,0.994900,-0.994901


,ID,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V22,V23,V24,V25,V26,V27,V28,V29,V30,Class
0,10,-0.338262,1.119593,1.044367,-0.222187,0.499361,-0.246761,0.651583,0.069539,-0.736727,...,-0.633753,-0.120794,-0.385050,-0.069733,0.094199,0.246219,0.083076,-0.255991,-0.994878,0
1,22,0.962496,0.328461,-0.171479,2.109204,1.129566,1.696038,0.107712,0.521502,-1.191311,...,0.402492,-0.048508,-1.371866,0.390814,0.199964,0.016371,-0.014605,0.168937,-0.994784,0
2,63,1.145524,0.575068,0.194008,2.598192,-0.092210,-1.044430,0.531588,-0.241888,-0.896287,...,-0.119703,-0.076510,0.691320,0.633984,0.048741,-0.053192,0.016251,0.169496,-0.994502,0
3,69,0.927060,-0.323684,0.387585,0.544474,0.246787,1.650358,-0.427576,0.615371,0.226278,...,0.079359,0.096632,-0.992569,0.085096,0.377447,0.036096,-0.005960,0.331307,-0.994467,0
4,83,-3.005237,2.600138,1.483691,-2.418473,0.306326,-0.824575,2.065426,-1.829347,4.009259,...,-0.181268,-0.163747,0.515821,0.136318,0.460054,-0.251259,-1.105751,-0.287012,-0.994373,0


,ID,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,V29,V30
0,AAAA0x1,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,1.783274,-0.994983
1,AAAA0x2,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,-0.269825,-0.994983
2,AAAA0x5,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,0.670579,-0.994960
3,AAAA0x7,1.229658,0.141004,0.045371,1.202613,0.191881,0.272708,-0.005159,0.081213,0.464960,...,-0.167716,-0.270710,-0.154104,-0.780055,0.750137,-0.257237,0.034507,0.005168,-0.237686,-0.994937
4,AAAA0xc,0.384978,0.616109,-0.874300,-0.094019,2.924584,3.317027,0.470455,0.538247,-0.558895,...,0.049924,0.238422,0.009130,0.996710,-0.767315,-0.492208,0.042472,-0.054337,-0.167819,-0.994866


In [ ]:
# 'Class' 컬럼을 기준으로 val 데이터를 class 0과 class 1로 나누기
val_class_0 = val[val['Class'] == 0].drop('Class', axis=1)  # Class가 0인 데이터
val_class_1 = val[val['Class'] == 1].drop('Class', axis=1)  # Class가 1인 데이터

# 나누어진 데이터 확인
print(f"val_class_0 shape: {val_class_0.shape}")
print(f"val_class_1 shape: {val_class_1.shape}")

val_normal, val_fraud = val['Class'].value_counts()
val_contamination = val_fraud / val_normal
print(f'Validation contamination : [{val_contamination}]')


val_class_0 shape: (28432, 31)
val_class_1 shape: (30, 31)
Validation contamination : [0.0010551491277433877]


In [ ]:
train= train.drop(columns=['ID'])
val_x = val.drop(columns=['ID', 'Class'])  # 검증 데이터에서 ID와 Class 열을 제외한 특성만 사용
val_y = val['Class']
print(f" train: {train.shape}")
print(f"val_x shape: {val_x.shape}")

display(train.head(), val_x.head(), val_y.head())

 train: (113842, 30)
val_x shape: (28462, 30)


,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V21,V22,V23,V24,V25,V26,V27,V28,V29,V30
0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,4.983721,-0.994972
1,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,1.418291,-0.994972
2,-0.425966,0.960523,1.141109,-0.168252,0.420987,-0.029728,0.476201,0.260314,-0.568671,-0.371407,...,-0.208254,-0.559825,-0.026398,-0.371427,-0.232794,0.105915,0.253844,0.081080,-0.256131,-0.994960
3,-0.644269,1.417964,1.074380,-0.492199,0.948934,0.428118,1.120631,-3.807864,0.615375,1.249376,...,1.943465,-1.015455,0.057504,-0.649709,-0.415267,-0.051634,-1.206921,-1.085339,0.262698,-0.994901
4,-0.894286,0.286157,-0.113192,-0.271526,2.669599,3.721818,0.370145,0.851084,-0.392048,-0.410430,...,-0.073425,-0.268092,-0.204233,1.011592,0.373205,-0.384157,0.011747,0.142404,0.994900,-0.994901


,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V21,V22,V23,V24,V25,V26,V27,V28,V29,V30
0,-0.338262,1.119593,1.044367,-0.222187,0.499361,-0.246761,0.651583,0.069539,-0.736727,-0.366846,...,-0.246914,-0.633753,-0.120794,-0.385050,-0.069733,0.094199,0.246219,0.083076,-0.255991,-0.994878
1,0.962496,0.328461,-0.171479,2.109204,1.129566,1.696038,0.107712,0.521502,-1.191311,0.724396,...,0.143997,0.402492,-0.048508,-1.371866,0.390814,0.199964,0.016371,-0.014605,0.168937,-0.994784
2,1.145524,0.575068,0.194008,2.598192,-0.092210,-1.044430,0.531588,-0.241888,-0.896287,0.757952,...,0.011106,-0.119703,-0.076510,0.691320,0.633984,0.048741,-0.053192,0.016251,0.169496,-0.994502
3,0.927060,-0.323684,0.387585,0.544474,0.246787,1.650358,-0.427576,0.615371,0.226278,-0.225495,...,-0.040513,0.079359,0.096632,-0.992569,0.085096,0.377447,0.036096,-0.005960,0.331307,-0.994467
4,-3.005237,2.600138,1.483691,-2.418473,0.306326,-0.824575,2.065426,-1.829347,4.009259,6.051521,...,-0.852309,-0.181268,-0.163747,0.515821,0.136318,0.460054,-0.251259,-1.105751,-0.287012,-0.994373


,Class
0,0
1,0
2,0
3,0
4,0


In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import classification_report

class AnomalyDetectionEnsemble:
    def __init__(self, contamination_ratio=None, nu_ratio=None, random_state=42):
        """
        Anomaly Detection Ensemble 모델 초기화
        :param contamination_ratio: IsolationForest 모델의 contamination 비율
        :param nu_ratio: One-Class SVM 모델의 nu 비율
        :param random_state: 랜덤 시드 값
        """
        self.contamination_ratio = contamination_ratio
        self.nu_ratio = nu_ratio
        self.random_state = random_state

        # 모델 초기화
        self.iso_forest = IsolationForest(contamination=self.contamination_ratio, random_state=self.random_state)
        self.one_class_svm = OneClassSVM(nu=self.nu_ratio, kernel='rbf', gamma='scale')

    def fit(self, train_data):
        """
        모델 학습
        :param train_data: 학습 데이터
        """
        self.iso_forest.fit(train_data)
        self.one_class_svm.fit(train_data)

    def predict(self, data):
        """
        예측 수행
        :param data: 예측할 데이터
        :return: 앙상블 예측 결과 (정상: 1, 이상치: -1)
        """
        # 각각의 모델 예측
        y_pred_iso = self.iso_forest.predict(data)  # 정상: 1, 이상치: -1
        y_pred_svm = self.one_class_svm.predict(data)  # 정상: 1, 이상치: -1

        # 앙상블 예측
        # ensemble_pred = np.mean([y_pred_iso, y_pred_svm], axis=0)
        ensemble_pred = np.average([y_pred_iso, y_pred_svm], axis=0, weights=[0.7, 0.3])
        ensemble_pred = np.where(ensemble_pred > 0, 1, -1)  # 예측 값이 0보다 크면 정상(1), 아니면 이상치(-1)

        return ensemble_pred

    def evaluate(self, true_labels, predictions):
        """
        모델 성능 평가
        :param true_labels: 실제 레이블 (정상: 0, 사기: 1)
        :param predictions: 예측된 레이블 (정상: 1, 이상치: -1)
        :return: classification_report
        """
        return classification_report(true_labels, predictions)

# 이상치 비율
contamination_ratio = 0.0010551491277433877
nu_ratio = 0.0010551491277433877

# 학습 및 예측
ensemble_model = AnomalyDetectionEnsemble(contamination_ratio=contamination_ratio, nu_ratio=nu_ratio)
ensemble_model.fit(train)

# 예측
ensemble_pred = ensemble_model.predict(val_x)



In [ ]:
ensemble_pred = np.where(ensemble_pred == -1, 1, 0)
val_score = f1_score(val_y, ensemble_pred, average='macro')
# 예측된 ensemble_pred 값의 첫 10개 출력
print(ensemble_pred[:100])
# 예측된 val_y 값의 첫 10개 출력
print(val_y[:100])

print(f'Validation F1 Score : [{val_score}]')

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
0     0
1     0
2     0
3     0
4     0
     ..
95    0
96    0
97    0
98    0
99    0
Name: Class, Length: 100, dtype: int64
Validation F1 Score : [0.6751132760065884]


In [ ]:
# 사용할 열 정의
selected_columns = ['V2', 'V3', 'V4', 'V5', 'V7', 'V8', 'V9', 'V10',
                    'V11', 'V12', 'V14', 'V16', 'V17', 'V18', 'V21', 'V27', 'V28']

# ID와 선택한 열만 남기고 나머지 제거
train_data = train[selected_columns]
val_x = val[selected_columns]
val_y = val['Class']
print(f" train shape: {train_data.shape}")
print(f"val_x shape: {val_x.shape}")

display(train_data.head(), val_x.head(), val_y.head())



In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import classification_report, f1_score, confusion_matrix

class AnomalyDetectionEnsemble:
    def __init__(self, contamination_ratio=None, nu_ratio=None, random_state=42):
        """
        Anomaly Detection Ensemble 모델 초기화
        :param contamination_ratio: IsolationForest 모델의 contamination 비율
        :param nu_ratio: One-Class SVM 모델의 nu 비율
        :param random_state: 랜덤 시드 값
        """
        self.contamination_ratio = contamination_ratio
        self.nu_ratio = nu_ratio
        self.random_state = random_state

        # 모델 초기화
        self.iso_forest = IsolationForest(contamination=self.contamination_ratio, random_state=self.random_state)
        self.one_class_svm = OneClassSVM(nu=self.nu_ratio, kernel='rbf', gamma='scale')

    def fit(self, train_data):
        """
        모델 학습
        :param train_data: 학습 데이터
        """
        self.iso_forest.fit(train_data)
        self.one_class_svm.fit(train_data)

    def predict(self, data):
        """
        예측 수행
        :param data: 예측할 데이터
        :return: 앙상블 예측 결과 (정상: 1, 이상치: -1)
        """
        # 각각의 모델 예측
        y_pred_iso = self.iso_forest.predict(data)  # 정상: 1, 이상치: -1
        y_pred_svm = self.one_class_svm.predict(data)  # 정상: 1, 이상치: -1

        # 앙상블 예측
        ensemble_pred = np.average([y_pred_iso, y_pred_svm], axis=0, weights=[0.7, 0.3])
        ensemble_pred = np.where(ensemble_pred > 0, 1, -1)  # 예측 값이 0보다 크면 정상(1), 아니면 이상치(-1)

        return ensemble_pred

    def evaluate(self, true_labels, predictions):
        """
        모델 성능 평가
        :param true_labels: 실제 레이블 (정상: 0, 사기: 1)
        :param predictions: 예측된 레이블 (정상: 1, 이상치: -1)
        :return: classification_report
        """
        return classification_report(true_labels, predictions)


# 이상치 비율
contamination_ratio = 0.0010551491277433877
nu_ratio = 0.0010551491277433877

# 학습 및 예측
ensemble_model = AnomalyDetectionEnsemble(contamination_ratio=contamination_ratio, nu_ratio=nu_ratio)
ensemble_model.fit(train_data)

# 예측
ensemble_pred = ensemble_model.predict(val_x)
ensemble_pred = np.where(ensemble_pred == -1, 1, 0)

# F1 Score 계산
val_score = f1_score(val_y, ensemble_pred, average='macro')
print(f'Validation F1 Score : [{val_score}]')

# 성능 평가
print(classification_report(val_y, ensemble_pred))

# 혼동 행렬 출력
tn, fp, fn, tp = confusion_matrix(val_y, ensemble_pred).ravel()
print('tp : ', tp, ', fp : ', fp, ', tn : ', tn, ', fn : ', fn)


Validation F1 Score : [0.6961296335373145]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     28432
           1       0.42      0.37      0.39        30

    accuracy                           1.00     28462
   macro avg       0.71      0.68      0.70     28462
weighted avg       1.00      1.00      1.00     28462

tp :  11 , fp :  15 , tn :  28417 , fn :  19
